# 05 — Causal Parameter Identification for MMM

## Objective

Use data-driven methods to identify optimal transformation parameters for Marketing Mix Modeling.

**This notebook answers:**
- What adstock decay rates does OUR data support? (not industry benchmarks)
- Which saturation functional form (Hill, Log, Power) fits best per channel?
- What are the optimal half-saturation points (K) for each channel?

**Approach:**
- Grid search over decay rates lambda in [0.1, 0.2, ..., 0.8]
- Test 3 saturation functions (Hill, Log, Power)
- LOOCV R-squared computed via `cross_val_predict` (correct method for LOO)
- Bootstrap confidence intervals on optimal parameters

**Data:**
- 36 monthly observations (FY2023-FY2025)
- 7 media channels (already aggregated from NB02)
- Target: total_all_revenue ($512M over 3 years)
- Controls: Fourier seasonality (2 harmonics) + 3 weather features
- Weather: Greater Montreal Area only (sunshine, precipitation, extreme heat)
- No holiday features (redundant with seasonality controls)

**Outputs:**
- `optimal_transformation_params.json` -- Calibrated parameters for use in notebook 06
- Visualizations showing grid search results
- Report documenting findings

---

## Section 1: Setup & Data Loading

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
from datetime import datetime

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

# Import our transformation utilities
import sys
sys.path.append(str(Path().cwd().parent))
from src.features.transformations import (
    geometric_adstock,
    hill_saturation,
    log_saturation,
    power_saturation,
    calculate_half_life,
    calculate_steady_state_gain
)

warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10
sns.set_palette('husl')

# Paths
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
processed_path = project_root / 'data' / 'processed'
figures_path = project_root / 'reports' / 'figures'
reports_path = project_root / 'reports'

figures_path.mkdir(parents=True, exist_ok=True)
reports_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed data: {processed_path}")
print(f"Figures: {figures_path}")
print(f"Reports: {reports_path}")

In [ ]:
# Load data (36 months of sales + spend + weather)
df = pd.read_pickle(processed_path / 'sales_spend_weather.pkl')
df = df.sort_values('date').reset_index(drop=True)

print(f"Dataset loaded: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Fiscal years: {sorted(df['year'].unique())}")
print(f"\nRevenue summary:")
rev_cols = [c for c in df.columns if 'revenue' in c]
for col in rev_cols:
    print(f"  {col:35s}  ${df[col].sum():>14,.0f}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Define 7 media channels (already aggregated in NB02)
SPEND_TO_MEDIA = {
    'media_television':          'spend_television',
    'media_radio':               'spend_radio',
    'media_panneaux':            'spend_panneaux',
    'media_social_media':        'spend_social_media',
    'media_preroll':             'spend_preroll',
    'media_banniere_web':        'spend_banniere_web',
    'media_circulaire_digitale': 'spend_circulaire_digitale',
}

# Industry benchmark decay rates (fallback when data can't discriminate)
INDUSTRY_BENCHMARKS = {
    'media_television':          0.7,   # TV ads have long memory
    'media_radio':               0.5,   # Audio has moderate carry-over
    'media_panneaux':            0.4,   # Outdoor/billboard mid-range
    'media_social_media':        0.4,   # Social has moderate decay
    'media_preroll':             0.5,   # Video pre-roll similar to TV but shorter
    'media_banniere_web':        0.3,   # Display ads decay faster
    'media_circulaire_digitale': 0.3,   # Flyers/circulars decay fast
}

# Create media columns (rename spend_ -> media_)
MEDIA_CHANNELS = []
for media_col, spend_col in SPEND_TO_MEDIA.items():
    df[media_col] = df[spend_col].fillna(0)
    MEDIA_CHANNELS.append(media_col)

df['media_total'] = df[MEDIA_CHANNELS].sum(axis=1)

print("Media channels configured:")
for ch in MEDIA_CHANNELS:
    total = df[ch].sum()
    pct = total / df['media_total'].sum() * 100
    nz = (df[ch] > 0).sum()
    print(f"  {ch:35s}  ${total:>12,.0f}  ({pct:5.1f}%)  non-zero: {nz}/{len(df)}")
print(f"  {'TOTAL':35s}  ${df['media_total'].sum():>12,.0f}")

In [ ]:
# Control features for the grid search models
# Fourier seasonality (2 harmonics to capture sharp spring peak)
df['sin_1'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['cos_1'] = np.cos(2 * np.pi * df['month_num'] / 12)
df['sin_2'] = np.sin(4 * np.pi * df['month_num'] / 12)
df['cos_2'] = np.cos(4 * np.pi * df['month_num'] / 12)

# Weather — only 3 strongest features (Greater Montreal Area)
# Selected based on de-seasonalized correlation analysis in NB03:
#   1. total_sunshine_hours — strongest residual signal
#   2. total_precipitation — orthogonal to temperature
#   3. days_above_25 — nonlinear extreme heat effect
for wcol in ['total_sunshine_hours', 'total_precipitation', 'days_above_25']:
    wmean = df[wcol].mean()
    wstd = df[wcol].std()
    df[f'{wcol}_scaled'] = (df[wcol] - wmean) / wstd

# NO holiday features — redundant with Fourier seasonality controls

CONTROL_COLS = ['sin_1', 'cos_1', 'sin_2', 'cos_2',
                'total_sunshine_hours_scaled', 'total_precipitation_scaled',
                'days_above_25_scaled']

# Primary target for parameter calibration
TARGET = 'total_all_revenue'

print(f"Control features ({len(CONTROL_COLS)}):")
for f in CONTROL_COLS:
    print(f"  {f}")
print(f"\nPrimary target: {TARGET}")
print(f"Observations: {len(df)}")
print(f"Features per grid search model: 1 media + {len(CONTROL_COLS)} controls = {1 + len(CONTROL_COLS)}")
print(f"Obs-to-feature ratio: {len(df) / (1 + len(CONTROL_COLS)):.1f}:1")

---
## Section 2: Adstock Decay Rate Calibration

**Goal:** Find the decay rate λ that maximizes LOOCV R² for each channel.

**Method:**
- For each channel and each λ ∈ [0.1, 0.2, ..., 0.8]:
  - Apply adstock with that decay rate
  - Fit Ridge model: total_all_revenue ~ adstocked_media + controls
  - Compute LOOCV R² using `cross_val_predict` (collects all LOO predictions, then computes R² once)
- Select λ that maximizes R²

**Fix from previous version:** The old code used `cross_val_score` with R² scoring, which returns NaN for LOO (single-sample R² is undefined). Now using `cross_val_predict` to collect all predictions first.

In [ ]:
# Grid search for optimal decay rates per channel
DECAY_GRID = np.arange(0.1, 0.9, 0.1)  # [0.1, 0.2, ..., 0.8]
ALPHA_RIDGE = 10.0  # Fixed regularization for parameter search

print(f"Grid search: {len(DECAY_GRID)} decay rates x {len(MEDIA_CHANNELS)} channels")
print(f"Target: {TARGET}")
print(f"Method: LOOCV with cross_val_predict -> R²")
print("\nRunning grid search...")

decay_results = []
y = df[TARGET].values

for channel in MEDIA_CHANNELS:
    print(f"\n{channel}:")

    for decay in DECAY_GRID:
        # Apply adstock with this decay rate
        adstock = geometric_adstock(df[channel].fillna(0).values, decay)

        # Build feature matrix: adstocked media + controls
        X = np.column_stack([
            adstock,
            df[CONTROL_COLS].fillna(0).values
        ])

        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # LOOCV: collect ALL predictions, then compute R² once
        model = Ridge(alpha=ALPHA_RIDGE)
        loo = LeaveOneOut()
        y_pred_cv = cross_val_predict(model, X_scaled, y, cv=loo)
        r2_cv = r2_score(y, y_pred_cv)

        decay_results.append({
            'channel': channel,
            'decay_rate': round(decay, 1),
            'r2_cv': r2_cv,
        })

    # Show best for this channel
    ch_results = [r for r in decay_results if r['channel'] == channel]
    best = max(ch_results, key=lambda x: x['r2_cv'])
    print(f"  Best lambda = {best['decay_rate']:.1f} (LOOCV R² = {best['r2_cv']:.4f})")

decay_df = pd.DataFrame(decay_results)
print("\nGrid search complete!")

In [ ]:
# Select optimal decay rate per channel
# Use data-driven result if R² improves over controls-only baseline; otherwise fall back to industry
OPTIMAL_DECAY_RATES = {}
DATA_DRIVEN_FLAGS = {}

# Compute controls-only baseline R² (no media)
X_ctrl = df[CONTROL_COLS].fillna(0).values
scaler_ctrl = StandardScaler()
X_ctrl_scaled = scaler_ctrl.fit_transform(X_ctrl)
y_pred_baseline = cross_val_predict(Ridge(alpha=ALPHA_RIDGE), X_ctrl_scaled, y, cv=LeaveOneOut())
r2_baseline = r2_score(y, y_pred_baseline)
print(f"Controls-only baseline LOOCV R²: {r2_baseline:.4f}\n")

print(f"{'Channel':30s}  {'lambda':>6s}  {'R² (CV)':>8s}  {'vs Base':>8s}  {'Source':15s}  {'Half-life':>10s}")
print("-" * 90)

for channel in MEDIA_CHANNELS:
    ch_results = decay_df[decay_df['channel'] == channel]
    best_row = ch_results.loc[ch_results['r2_cv'].idxmax()]
    best_decay = float(best_row['decay_rate'])
    best_r2 = float(best_row['r2_cv'])
    r2_improvement = best_r2 - r2_baseline

    # Use data-driven if it improves over baseline by at least 0.01
    if r2_improvement > 0.01:
        OPTIMAL_DECAY_RATES[channel] = best_decay
        DATA_DRIVEN_FLAGS[channel] = True
        source = "Data-driven"
    else:
        OPTIMAL_DECAY_RATES[channel] = INDUSTRY_BENCHMARKS[channel]
        DATA_DRIVEN_FLAGS[channel] = False
        source = "Industry bench."
        best_decay = INDUSTRY_BENCHMARKS[channel]

    half_life = calculate_half_life(best_decay)
    print(f"{channel:30s}  {best_decay:>6.1f}  {best_r2:>8.4f}  {r2_improvement:>+8.4f}  {source:15s}  {half_life:>8.1f}mo")

print("-" * 90)
n_data = sum(DATA_DRIVEN_FLAGS.values())
n_fallback = len(DATA_DRIVEN_FLAGS) - n_data
print(f"\n{n_data} channels data-driven, {n_fallback} using industry benchmarks")

In [ ]:
# Visualize grid search results
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, channel in enumerate(MEDIA_CHANNELS):
    ax = axes[idx]
    ch_results = decay_df[decay_df['channel'] == channel]

    ax.plot(ch_results['decay_rate'], ch_results['r2_cv'],
            'o-', linewidth=2, markersize=8, color='steelblue')

    # Mark optimal
    optimal = OPTIMAL_DECAY_RATES[channel]
    match = ch_results[(ch_results['decay_rate'] - optimal).abs() < 0.05]
    if len(match) > 0:
        ax.scatter([optimal], [match['r2_cv'].values[0]], color='red', s=200,
                   marker='*', zorder=5, edgecolors='black')

    # Baseline
    ax.axhline(r2_baseline, color='grey', linestyle='--', linewidth=0.8,
               label=f'Baseline R²={r2_baseline:.3f}')

    source = "data" if DATA_DRIVEN_FLAGS.get(channel, False) else "benchmark"
    ch_label = channel.replace('media_', '').replace('_', ' ').title()
    ax.set_xlabel('Decay Rate (lambda)')
    ax.set_ylabel('LOOCV R²')
    ax.set_title(f"{ch_label} ({source}, lambda={optimal:.1f})")
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

fig.delaxes(axes[-1])
fig.suptitle('Adstock Decay Rate Grid Search — LOOCV R² vs total_all_revenue\n'
             '(Red star = selected optimal)', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(figures_path / 'causal_decay_grid_search.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: Saturation Function Comparison

**Goal:** Test which saturation function (Hill, Log, Power) works best per channel.

**Method:**
- Use optimal decay rate from Section 2
- Apply adstock
- Test 3 saturation functions:
  - Hill: with K = median, α = 2
  - Log: with scale = median
  - Power: with β = 0.5
- Compare LOOCV R²

In [ ]:
# Compare saturation functions (Hill, Log, Power) per channel
saturation_results = []

print("Testing saturation functions per channel (using optimal decay rates)...\n")

for channel in MEDIA_CHANNELS:
    print(f"{channel}:")

    decay = OPTIMAL_DECAY_RATES[channel]
    adstock = geometric_adstock(df[channel].fillna(0).values, decay)
    K_median = np.median(adstock[adstock > 0]) if np.any(adstock > 0) else 1.0

    for sat_name, sat_func in [
        ('hill', lambda x, K=K_median: hill_saturation(x, K, alpha=2)),
        ('log', lambda x, K=K_median: log_saturation(x, scale=K)),
        ('power', lambda x: power_saturation(x, beta=0.5)),
    ]:
        saturated = sat_func(adstock)

        X = np.column_stack([saturated, df[CONTROL_COLS].fillna(0).values])
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        y_pred_cv = cross_val_predict(Ridge(alpha=ALPHA_RIDGE), X_scaled, y, cv=LeaveOneOut())
        r2_cv = r2_score(y, y_pred_cv)

        saturation_results.append({
            'channel': channel,
            'saturation_func': sat_name,
            'r2_cv': r2_cv,
        })
        print(f"  {sat_name:6s}: LOOCV R² = {r2_cv:.4f}")

saturation_df = pd.DataFrame(saturation_results)
print("\nSaturation function comparison complete!")

In [ ]:
# Select optimal saturation function per channel
OPTIMAL_SATURATION_FUNCS = {}

print("Optimal saturation functions:")
print("=" * 70)
for channel in MEDIA_CHANNELS:
    ch_results = saturation_df[saturation_df['channel'] == channel]
    best_row = ch_results.loc[ch_results['r2_cv'].idxmax()]
    best_func = best_row['saturation_func']
    best_r2 = float(best_row['r2_cv'])

    OPTIMAL_SATURATION_FUNCS[channel] = best_func
    ch_label = channel.replace('media_', '').replace('_', ' ').title()
    print(f"{ch_label:30s}  {best_func:6s}  R²={best_r2:.4f}")

print("=" * 70)

In [ ]:
# Visualize saturation function comparison
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(MEDIA_CHANNELS))
width = 0.25
colors = {'hill': '#2ecc71', 'log': '#3498db', 'power': '#e74c3c'}

for i, func in enumerate(['hill', 'log', 'power']):
    r2_values = [
        saturation_df[
            (saturation_df['channel'] == ch) &
            (saturation_df['saturation_func'] == func)
        ]['r2_cv'].values[0]
        for ch in MEDIA_CHANNELS
    ]
    ax.bar(x + i*width, r2_values, width, label=func.capitalize(),
           alpha=0.8, color=colors[func])

ax.axhline(r2_baseline, color='grey', linestyle='--', linewidth=0.8,
           label=f'Controls-only baseline ({r2_baseline:.3f})')
ax.set_xlabel('Channel')
ax.set_ylabel('LOOCV R²')
ax.set_title('Saturation Function Comparison by Channel (total_all_revenue)')
ax.set_xticks(x + width)
ax.set_xticklabels([ch.replace('media_', '').replace('_', '\n') for ch in MEDIA_CHANNELS])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / 'causal_saturation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4: Half-Saturation Point (K) Calibration

**Goal:** For channels using Hill function, find optimal K.

**Method:**
- Test K values at different percentiles of adstocked spend
- Select K that maximizes LOOCV R²

In [ ]:
# Calibrate K (half-saturation) for Hill-function channels
hill_channels = [ch for ch in MEDIA_CHANNELS if OPTIMAL_SATURATION_FUNCS[ch] == 'hill']
K_results = []
OPTIMAL_K_VALUES = {}

print(f"Calibrating K for {len(hill_channels)} Hill-function channels...\n")

for channel in hill_channels:
    decay = OPTIMAL_DECAY_RATES[channel]
    adstock = geometric_adstock(df[channel].fillna(0).values, decay)

    adstock_nonzero = adstock[adstock > 0]
    if len(adstock_nonzero) == 0:
        OPTIMAL_K_VALUES[channel] = 1.0
        print(f"  {channel}: No non-zero spend, using K=1.0")
        continue

    K_candidates = np.percentile(adstock_nonzero, [10, 25, 50, 75, 90])
    best_r2_k = -np.inf
    best_K = float(np.median(adstock_nonzero))

    for K in K_candidates:
        saturated = hill_saturation(adstock, K, alpha=2)
        X = np.column_stack([saturated, df[CONTROL_COLS].fillna(0).values])
        X_scaled = StandardScaler().fit_transform(X)

        y_pred_cv = cross_val_predict(Ridge(alpha=ALPHA_RIDGE), X_scaled, y, cv=LeaveOneOut())
        r2_cv = r2_score(y, y_pred_cv)

        K_results.append({'channel': channel, 'K': K, 'r2_cv': r2_cv})

        if r2_cv > best_r2_k:
            best_r2_k = r2_cv
            best_K = float(K)

    OPTIMAL_K_VALUES[channel] = best_K
    ch_label = channel.replace('media_', '').replace('_', ' ').title()
    print(f"  {ch_label:30s}  K=${best_K:>12,.0f}  R²={best_r2_k:.4f}")

# For non-Hill channels, store appropriate parameters
for channel in MEDIA_CHANNELS:
    if channel not in OPTIMAL_K_VALUES:
        decay = OPTIMAL_DECAY_RATES[channel]
        adstock = geometric_adstock(df[channel].fillna(0).values, decay)
        if OPTIMAL_SATURATION_FUNCS[channel] == 'log':
            scale = float(np.median(adstock[adstock > 0])) if np.any(adstock > 0) else 1.0
            OPTIMAL_K_VALUES[channel] = scale
        else:
            OPTIMAL_K_VALUES[channel] = 0.5

print("\nK calibration complete!")

---
## Section 5: Bootstrap Confidence Intervals

Quantify uncertainty in optimal parameters via bootstrap.

In [ ]:
# Bootstrap confidence intervals on decay rates
N_BOOT = 200
np.random.seed(42)

print(f"Running bootstrap ({N_BOOT} resamples) for confidence intervals...\n")

boot_decay = {ch: [] for ch in MEDIA_CHANNELS}
n = len(df)

for b in range(N_BOOT):
    if b % 50 == 0:
        print(f"  Bootstrap iteration {b}/{N_BOOT}")

    idx = np.random.choice(n, size=n, replace=True)
    df_boot = df.iloc[idx].reset_index(drop=True)
    y_boot = df_boot[TARGET].values

    for channel in MEDIA_CHANNELS:
        optimal_decay = OPTIMAL_DECAY_RATES[channel]
        test_decays = np.clip([optimal_decay - 0.1, optimal_decay, optimal_decay + 0.1], 0.1, 0.8)

        best_r2 = -np.inf
        best_decay_boot = optimal_decay

        for decay in test_decays:
            adstock = geometric_adstock(df_boot[channel].fillna(0).values, decay)
            K_temp = np.median(adstock[adstock > 0]) if np.any(adstock > 0) else 1.0
            saturated = hill_saturation(adstock, K_temp, alpha=2)

            X = np.column_stack([saturated, df_boot[CONTROL_COLS].fillna(0).values])
            X_scaled = StandardScaler().fit_transform(X)

            model = Ridge(alpha=ALPHA_RIDGE).fit(X_scaled, y_boot)
            r2 = r2_score(y_boot, model.predict(X_scaled))

            if r2 > best_r2:
                best_r2 = r2
                best_decay_boot = decay

        boot_decay[channel].append(best_decay_boot)

# Confidence intervals
DECAY_CONFIDENCE_INTERVALS = {}
print(f"\nBootstrap 90% Confidence Intervals:")
print("=" * 70)
for channel in MEDIA_CHANNELS:
    boot_vals = np.array(boot_decay[channel])
    ci_lower = np.percentile(boot_vals, 5)
    ci_upper = np.percentile(boot_vals, 95)
    DECAY_CONFIDENCE_INTERVALS[channel] = (round(float(ci_lower), 2), round(float(ci_upper), 2))

    optimal = OPTIMAL_DECAY_RATES[channel]
    ch_label = channel.replace('media_', '').replace('_', ' ').title()
    print(f"  {ch_label:30s}  lambda={optimal:.1f}  CI=[{ci_lower:.2f}, {ci_upper:.2f}]")
print("=" * 70)

---
## Section 6: Save Results

Package all calibrated parameters into JSON for use in notebook 06.

In [ ]:
# Build parameter dict for NB06
params = {
    'decay_rates': OPTIMAL_DECAY_RATES,
    'decay_rate_sources': {ch: ('data' if DATA_DRIVEN_FLAGS.get(ch, False) else 'industry')
                           for ch in MEDIA_CHANNELS},
    'industry_benchmarks': INDUSTRY_BENCHMARKS,
    'saturation_functions': OPTIMAL_SATURATION_FUNCS,
    'saturation_params': {},
    'confidence_intervals': DECAY_CONFIDENCE_INTERVALS,
    'calibration_method': 'LOOCV_Ridge_cross_val_predict',
    'calibration_date': datetime.now().strftime('%Y-%m-%d'),
    'n_observations': len(df),
    'n_bootstrap_samples': N_BOOT,
    'target': TARGET,
    'control_features': CONTROL_COLS,
    'baseline_r2': round(r2_baseline, 4),
}

# Add saturation parameters per channel
for channel in MEDIA_CHANNELS:
    func = OPTIMAL_SATURATION_FUNCS[channel]
    if func == 'hill':
        params['saturation_params'][channel] = {
            'K': OPTIMAL_K_VALUES[channel],
            'alpha': 2
        }
    elif func == 'log':
        params['saturation_params'][channel] = {
            'scale': OPTIMAL_K_VALUES[channel]
        }
    else:
        params['saturation_params'][channel] = {
            'beta': 0.5
        }

# Save to JSON
output_path = processed_path / 'optimal_transformation_params.json'
with open(output_path, 'w') as f:
    json.dump(params, f, indent=2)

print(f"Saved optimal parameters to: {output_path}")
print(f"\nSummary:")
print(f"  - {len(MEDIA_CHANNELS)} channels calibrated")
print(f"  - {sum(DATA_DRIVEN_FLAGS.values())} data-driven, "
      f"{len(DATA_DRIVEN_FLAGS) - sum(DATA_DRIVEN_FLAGS.values())} industry benchmarks")
print(f"  - Saturation functions: {dict(pd.Series(list(OPTIMAL_SATURATION_FUNCS.values())).value_counts())}")
print(f"  - Baseline R²: {r2_baseline:.4f}")
print(f"  - Bootstrap CIs: 90% (N={N_BOOT})")

---
## Section 7: Generate Report

Create markdown report summarizing findings.

In [ ]:
# Generate summary report
report_lines = [
    "# Feature Engineering & Parameter Calibration Report",
    f"\nGenerated: {datetime.now()}",
    f"\n{len(df)} monthly observations, {len(MEDIA_CHANNELS)} channels, target: {TARGET}",
    f"\nControls-only baseline LOOCV R²: {r2_baseline:.4f}",
    "\n## Decay Rates",
    "\n| Channel | Lambda | Source | CI | Half-life |",
    "|---|---|---|---|---|",
]

for channel in MEDIA_CHANNELS:
    ch_name = channel.replace('media_', '').replace('_', ' ').title()
    decay = OPTIMAL_DECAY_RATES[channel]
    source = "data" if DATA_DRIVEN_FLAGS.get(channel, False) else "industry"
    ci = DECAY_CONFIDENCE_INTERVALS.get(channel, (0, 0))
    hl = calculate_half_life(decay)
    report_lines.append(f"| {ch_name} | {decay:.1f} | {source} | [{ci[0]:.2f}, {ci[1]:.2f}] | {hl:.1f}mo |")

report_lines.extend([
    "\n## Saturation Functions",
    "\n| Channel | Function | K / Scale |",
    "|---|---|---|",
])
for channel in MEDIA_CHANNELS:
    ch_name = channel.replace('media_', '').replace('_', ' ').title()
    func = OPTIMAL_SATURATION_FUNCS[channel]
    K = OPTIMAL_K_VALUES.get(channel, 'N/A')
    if isinstance(K, (int, float)):
        report_lines.append(f"| {ch_name} | {func} | ${K:,.0f} |")
    else:
        report_lines.append(f"| {ch_name} | {func} | {K} |")

report_text = '\n'.join(report_lines)

report_path = reports_path / 'causal_identification_report.md'
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"Saved report to: {report_path}")
print("\n" + "=" * 70)
print(report_text)
print("=" * 70)

---
## Conclusion

**Parameter calibration complete.**

**Outputs:**
1. `optimal_transformation_params.json` — calibrated decay rates, saturation functions, and K values
2. `causal_decay_grid_search.png` — grid search visualization
3. `causal_saturation_comparison.png` — saturation function comparison
4. `causal_identification_report.md` — summary report

**Key improvements over previous version:**
- Fixed LOOCV R² computation (was returning NaN due to single-sample R²)
- 36 monthly observations (up from 19) — nearly 2x the statistical power
- Revenue-based target (`total_all_revenue`) instead of quote counts
- Reduced weather features to 3 strongest (sunshine, precipitation, hot days)
- Dropped redundant holiday features
- Controls-only baseline used as threshold for data-driven vs industry benchmark selection

**Next step:** Notebook 06 will load these parameters and run the causal inference models.